In [1]:
import pandas as pd

df = pd.read_csv('../data/cosmetics.csv')
print("Loaded", len(df), "products")

Loaded 1472 products


In [2]:
# Common skincare allergens - this is your knowledge base
ALLERGENS = [
    'fragrance', 'parfum', 'alcohol', 'parabens', 'methylparaben',
    'propylparaben', 'formaldehyde', 'lanolin', 'nickel', 'gluten',
    'sulfates', 'sodium lauryl sulfate', 'sls', 'phthalates',
    'oxybenzone', 'retinol', 'salicylic acid', 'glycolic acid',
    'linalool', 'limonene', 'geraniol', 'citronellol', 'benzyl salicylate'
]

print(f"Tracking {len(ALLERGENS)} allergens")

Tracking 23 allergens


In [3]:
def parse_ingredients(ingredient_string):
    """Split raw ingredient string into a clean list"""
    if pd.isna(ingredient_string):
        return []
    ingredients = [i.strip().lower() for i in ingredient_string.split(',')]
    return ingredients

def check_allergens(ingredient_string, user_allergens):
    """Check which of the user's allergens are in a product"""
    ingredients = parse_ingredients(ingredient_string)
    found = []
    for allergen in user_allergens:
        for ingredient in ingredients:
            if allergen.lower() in ingredient:
                found.append(allergen)
                break
    return found

In [4]:
# Simulate a user who is allergic to fragrance and alcohol
user_allergens = ['fragrance', 'alcohol', 'linalool']

# Test on the first product
product = df.iloc[0]
print("Product:", product['Name'])
print("Brand:", product['Brand'])
print()

allergens_found = check_allergens(product['Ingredients'], user_allergens)

if allergens_found:
    print("WARNING - contains your allergens:", allergens_found)
else:
    print("Safe for you - no allergens found")

Product: Crème de la Mer
Brand: LA MER

WARNING - contains your allergens: ['fragrance', 'alcohol', 'linalool']


In [5]:
# Flag every product for this user's allergens
df['allergens_found'] = df['Ingredients'].apply(
    lambda x: check_allergens(x, user_allergens)
)
df['is_safe'] = df['allergens_found'].apply(lambda x: len(x) == 0)

print(f"Total products: {len(df)}")
print(f"Safe for user: {df['is_safe'].sum()}")
print(f"Contains allergens: {(~df['is_safe']).sum()}")

Total products: 1472
Safe for user: 565
Contains allergens: 907


In [6]:
safe_products = df[df['is_safe']][['Name', 'Brand', 'Label', 'Price']].head(10)
print(safe_products.to_string())

                                                           Name           Brand        Label  Price
1                                      Facial Treatment Essence           SK-II  Moisturizer    179
4                 Your Skin But Better™ CC+™ Cream with SPF 50+    IT COSMETICS  Moisturizer     38
7                               Virgin Marula Luxury Facial Oil  DRUNK ELEPHANT  Moisturizer     72
13                                      Luna Sleeping Night Oil    SUNDAY RILEY  Moisturizer    105
17            Moisture Surge 72-Hour Auto-Replenishing Hydrator        CLINIQUE  Moisturizer     39
21  COMPLEXION RESCUE™ Tinted Moisturizer Broad Spectrum SPF 30    BAREMINERALS  Moisturizer     30
26                         Virgin Marula Luxury Facial Oil Mini  DRUNK ELEPHANT  Moisturizer     40
32                 Sheer Transformation® Perfecting Moisturizer    OLEHENRIKSEN  Moisturizer     38
33                                   100 percent Pure Argan Oil     JOSIE MARAN  Moisturizer     48
